# SituationCatch-Bench — 인간 IAA (Kaggle 실행용, self-contained)

이 노트북은 **채점 로직을 내부에 내장**하여 Kaggle에서 인터넷 연결이나 저장소 클론 없이도
완전히 동작합니다. 흐름:

1. **STEP A** — 빈 패킷/안내서 받기 (주석자에게 배부; 인터넷 On일 때만)
2. **STEP B** — 채운 `annotator_1/2/3.csv` 3개를 Kaggle에 업로드
3. **STEP C** — 자동 검증 (빈칸/허용값)
4. **STEP D** — Fleiss κ + 불일치표 계산
5. **STEP E** — 결과 파일(`/kaggle/working/`)을 Output 탭에서 내려받아 전달

셀을 위에서부터 `Shift+Enter`로 실행하세요.

## STEP B 준비 — Kaggle에서 파일 업로드하는 법 (중요)
Kaggle에는 Colab의 파일 업로드 창이 없습니다. 아래 중 **하나**로 3개 CSV를 올리세요.

- **방법 1 (권장): Add Input → Upload**
  1. 우측 상단 **Add Input**(또는 우측 패널 **+ Add Data**) 클릭 → **Upload** 탭
  2. `annotator_1.csv`, `annotator_2.csv`, `annotator_3.csv` 3개를 드래그 → 데이터셋 생성
  3. 그러면 `/kaggle/input/<데이터셋이름>/` 아래에 파일이 마운트됩니다.
- **방법 2: 노트북 파일 패널 Upload** → 파일이 `/kaggle/working/` 에 올라갑니다.

어디에 올리든 아래 STEP B 셀이 **자동으로 찾아냅니다.**

## STEP A — (선택) 빈 패킷/안내서 받기 — *인터넷 On* 필요
우측 **Settings → Internet: On** 이면 아래가 저장소에서 빈 패킷을 받아 zip으로 만듭니다.
Internet Off면 건너뛰고, 아래 GitHub 링크에서 직접 받으세요:
`https://github.com/leemgs/sage/tree/main/paper/annotation_packets`

In [ ]:
import os, shutil
REPO='https://github.com/leemgs/sage.git'
try:
    if not os.path.isdir('/kaggle/working/sage'):
        rc=os.system(f'git clone --depth 1 {REPO} /kaggle/working/sage')
    pk='/kaggle/working/sage/paper/annotation_packets'
    if os.path.isdir(pk):
        shutil.make_archive('/kaggle/working/annotation_packets_blank','zip',pk)
        print('OK: /kaggle/working/annotation_packets_blank.zip (Output 탭에서 다운로드해 배부)')
        print('안내서/코드북:')
        print(open(pk+'/HOW_TO_ANNOTATE.md',encoding='utf-8').read()[:1500],'...')
    else:
        raise RuntimeError('clone dir missing')
except Exception as e:
    print('(인터넷 Off이거나 클론 실패) 수동 다운로드하세요:')
    print('https://github.com/leemgs/sage/tree/main/paper/annotation_packets')

## STEP B — 업로드한 채운 파일 3개 자동 탐색

In [ ]:
import glob, os
pats=['/kaggle/input/**/annotator_*.csv','/kaggle/working/**/annotator_*.csv','annotator_*.csv']
found={}
for p in pats:
    for f in glob.glob(p, recursive=True):
        b=os.path.basename(f)
        if b.startswith('annotator_') and b not in found:  # EXAMPLE_* 제외됨
            found[b]=f
PACKETS=[found[b] for b in sorted(found)]
print('찾은 파일:')
for f in PACKETS: print('  ', f)
assert len(PACKETS)>=3, f'annotator_1/2/3.csv 3개가 필요합니다. 발견 {len(PACKETS)}개 — STEP B 준비 참고해 업로드하세요.'
PACKETS=PACKETS[:3]

## STEP C — 검증 (70행 · 빈칸 · 허용값)

In [ ]:
import csv
ALLOWED={
 'action':{'ANSWER','CLARIFY','ABSTAIN'},
 'temporal_state':{'relevant','stable'},
 'modality':{'confirmed','proposed'},
 'scope':{'global','limited'},
 'source_status':{'reliable','conflict'},
 'observer_state':{'shared','partial'},
 'world':{'actual','counterfactual'},
}
ok=True
for f in PACKETS:
    rows=list(csv.DictReader(open(f,encoding='utf-8-sig')))
    if len(rows)!=70: ok=False; print(f'[{os.path.basename(f)}] 70행 아님: {len(rows)}')
    for n,r in enumerate(rows,2):
        if not (r.get('answer') or '').strip(): ok=False; print(f'[{os.path.basename(f)}] {n}행 answer 비어있음')
        for col,vals in ALLOWED.items():
            v=(r.get(col) or '').strip()
            norm=v.upper() if col=='action' else v.lower()
            allow={x.upper() for x in vals} if col=='action' else vals
            if not v: ok=False; print(f'[{os.path.basename(f)}] {n}행 {col} 비어있음')
            elif norm not in allow: ok=False; print(f'[{os.path.basename(f)}] {n}행 {col}="{v}" 허용값 아님 {sorted(vals)}')
print('\n[OK] 통과: 다음 단계로.' if ok else '\n[!] 위 항목 수정 후 다시 업로드/실행하세요.')
assert ok, '검증 실패'

## STEP D — Fleiss κ + 불일치표 (내장 로직, 저장소 코드와 동일)
`RATING_SLOTS` 8개(action, answer, + 상태 6개)에 대해 슬롯별 일치도를 계산합니다.

In [ ]:
import json
from collections import Counter, defaultdict
RATING_SLOTS=['action','answer','temporal_state','modality','scope','source_status','observer_state','world']

def fleiss_kappa(labels_by_item):
    items=list(labels_by_item.values()); n=len(items[0])
    cats=sorted({x for labels in items for x in labels})
    p={c: sum(labels.count(c) for labels in items)/(len(items)*n) for c in cats}
    pbar=sum((sum(v*v for v in Counter(labels).values())-n)/(n*(n-1)) for labels in items)/len(items)
    pe=sum(v*v for v in p.values())
    return (pbar-pe)/(1-pe) if pe<1 else 1.0

# load packets, align item sets, read questions
packets=[]; expected=None; questions={}
for f in PACKETS:
    rows=list(csv.DictReader(open(f,encoding='utf-8-sig')))
    ids=[r['item_id'] for r in rows]
    assert len(ids)==len(set(ids)), f'중복 item_id: {f}'
    if expected is None: expected=set(ids)
    else: assert set(ids)==expected, f'문항 집합 불일치: {f}'
    for r in rows: questions[r['item_id']]=r.get('question','')
    packets.append(rows)

by_slot={s: defaultdict(list) for s in RATING_SLOTS}
for rows in packets:
    for r in rows:
        for s in RATING_SLOTS: by_slot[s][r['item_id']].append((r[s] or '').strip().casefold())
report={}
for s,items in by_slot.items():
    assert all(len(v)==len(packets) for v in items.values()), f'{s} 독립 평가 수 불일치'
    report[s]={'fleiss_kappa':fleiss_kappa(items),
               'unanimous_rate':sum(len(set(v))==1 for v in items.values())/len(items)}
agreement={'schema_version':1,'provenance':'human_annotations','synthetic':False,
           'n_annotators':len(packets),'n_items':len(expected),'slots':report}
os.makedirs('/kaggle/working',exist_ok=True)
json.dump(agreement, open('/kaggle/working/agreement.json','w'), indent=2, ensure_ascii=False)

# adjudication: 슬롯별 불일치 문항
adj=[]
rat=defaultdict(lambda: defaultdict(list))
for rows in packets:
    for r in rows:
        for s in RATING_SLOTS: rat[r['item_id']][s].append((r[s] or '').strip())
for iid,slots in rat.items():
    for s,vals in slots.items():
        if len({v.casefold() for v in vals})>1:
            adj.append({'item_id':iid,'question':questions[iid],'slot':s,
                        'independent_labels':json.dumps(vals,ensure_ascii=False),
                        'adjudicated_label':'','adjudicator_rationale':''})
with open('/kaggle/working/adjudication.csv','w',newline='',encoding='utf-8') as fp:
    w=csv.DictWriter(fp,fieldnames=['item_id','question','slot','independent_labels','adjudicated_label','adjudicator_rationale'])
    w.writeheader(); w.writerows(adj)

print(f"annotators={agreement['n_annotators']}  items={agreement['n_items']}  disagreements={len(adj)}")
print(f"{'slot':16s}{'fleiss_kappa':>14s}{'unanimous':>12s}")
for s,v in report.items(): print(f"{s:16s}{v['fleiss_kappa']:14.3f}{v['unanimous_rate']:12.2%}")

## STEP E — 결과 내려받아 전달
`/kaggle/working/` 의 파일은 노트북 저장 시 **Output**에 남습니다. 아래 셀이 zip으로 묶습니다.
우측 **Output** 탭(또는 파일 패널)에서 `human_iaa_results.zip` 을 다운로드해 저에게 주세요.

In [ ]:
import zipfile
with zipfile.ZipFile('/kaggle/working/human_iaa_results.zip','w') as z:
    z.write('/kaggle/working/agreement.json','agreement.json')
    z.write('/kaggle/working/adjudication.csv','adjudication.csv')
    for f in PACKETS: z.write(f, os.path.basename(f))
print('완료 → /kaggle/working/human_iaa_results.zip (Output 탭에서 다운로드)')
print('전달 파일: agreement.json, adjudication.csv, annotator_1/2/3.csv')